In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import re
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV, train_test_split, RepeatedKFold
from sklearn.inspection import PartialDependenceDisplay
from scipy.stats import randint, uniform

# 📁 文件夹路径
grid_folder = r'D:\seoul\grids\lst_map'

# 🔧 变量定义
target_vars = ['nor_2020', 'ext_2020', 'hr_2020']
explanatory_vars = ['BCR(%)', 'BHV', 'NDVI', 'SVF', 'EV(m)',
                    'Dist_BP', 'Dist_MT', 'Dist_WB', 'WR(%)']

# 📊 保存结果
all_results = []
pdp_records = []
r2_comparison = []

# 🔍 Randomized Search 的分布
param_dist = {
    # 森林中树的数量
    'n_estimators': randint(100, 1000),
    # 最大深度
    'max_depth': randint(5, 50),
    # 每次分割时节点所需的最少样本数
    'min_samples_split': randint(2, 11),
    # 叶节点最少样本数
    'min_samples_leaf': randint(1, 11),
    # 最大特征数（占比）
    'max_features': uniform(0.3, 0.7),
    # 样本采样（bootstrap）是否启用: True/False
    'bootstrap': [True, False]
}

# === 主循环 ===
for filename in os.listdir(grid_folder):
    if filename.endswith('_clean.shp'):
        input_path = os.path.join(grid_folder, filename)
        match = re.search(r'(\d{3,5})m', filename)
        grid_size = match.group(1)

        gdf = gpd.read_file(input_path)
        gdf_clean = gdf.replace([np.inf, -np.inf], np.nan).dropna(subset=target_vars + explanatory_vars)

        for target in target_vars:
            X = gdf_clean[explanatory_vars]
            y = gdf_clean[target]

            # 🔀 数据划分
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

            # 定义随机森林模型和交叉验证策略
            rf = RandomForestRegressor(random_state=0)
            cv = RepeatedKFold(n_splits=5, n_repeats=4, random_state=0)
            search = RandomizedSearchCV(
                estimator=rf,
                param_distributions=param_dist,
                n_iter=200,
                scoring='r2',
                cv=cv,
                verbose=2,
                n_jobs=-1,
                random_state=0
            )

            # 🔧 训练模型
            search.fit(X_train, y_train)

            # ✅ 使用测试集评估
            best_model = search.best_estimator_
            y_train_pred = best_model.predict(X_train)
            y_test_pred = best_model.predict(X_test)

            r2_train = best_model.score(X_train, y_train)
            r2_test = r2_score(y_test, y_test_pred)
            rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
            rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

            print(f"✅ {filename} | {target} 最佳参数: {search.best_params_} | R²_train={r2_train:.3f} | R²_test={r2_test:.3f}")

            # 记录结果
            for var, importance in zip(explanatory_vars, best_model.feature_importances_):
                all_results.append({
                    'GridSize': grid_size,
                    'Target': target,
                    'Feature': var,
                    'FeatureImportance_TrainModel': round(importance, 4),
                    'Train_R2': round(r2_train, 4),
                    'Train_RMSE': round(rmse_train, 4),
                    'Test_R2': round(r2_test, 4),
                    'Test_RMSE': round(rmse_test, 4),
                    **search.best_params_
                })
                r2_comparison.append({
                    'GridSize': grid_size,
                    'Target': target,
                    'Train_R2': round(r2_train, 4),
                    'Test_R2': round(r2_test, 4),
                    'Train_RMSE': round(rmse_train, 4),
                    'Test_RMSE': round(rmse_test, 4)
                })

            # 📈 PDP 提取（Top N 变量）
            sorted_idx = np.argsort(best_model.feature_importances_)[::-1]
            top_features = [explanatory_vars[i] for i in sorted_idx[:9]]

            for feature in top_features:
                fig, ax = plt.subplots()
                disp = PartialDependenceDisplay.from_estimator(best_model, X, [feature], ax=ax)
                x_vals = disp.lines_[0][0].get_xdata()
                y_vals = disp.lines_[0][0].get_ydata()
                plt.close(fig)
                pdp_records.append({
                    'Feature': feature,
                    'GridSize': grid_size,
                    'Target': target,
                    'X': x_vals,
                    'Y': y_vals
                })

# 保存结果
pd.DataFrame(all_results).to_excel(os.path.join(grid_folder, 'RF_Random_Search_Results.xlsx'), index=False)
pd.DataFrame(r2_comparison).sort_values(['Target', 'GridSize']).to_excel(os.path.join(grid_folder, 'RF_R2_Comparison_Train_vs_Test.xlsx'), index=False)
print("✅ R² train vs test comparison saved.")

# 保存 PDP 数据
pdp_df = pd.DataFrame(pdp_records)
pdp_df.to_pickle(os.path.join(grid_folder, 'rf_pdp_records.pkl'))  # 用 pickle 保留 numpy 数组

Fitting 20 folds for each of 200 candidates, totalling 4000 fits


C:\Users\owner\Python\Python311\Lib\site-packages\numpy\ma\core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


✅ city2020_lst_ratio_grid_1080m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp | nor_2020 最佳参数: {'bootstrap': False, 'max_depth': 45, 'max_features': 0.425643182642499, 'min_samples_leaf': 1, 'min_samples_split': 4, 'n_estimators': 297} | R²_train=0.999 | R²_test=0.872
Fitting 20 folds for each of 200 candidates, totalling 4000 fits
✅ city2020_lst_ratio_grid_1080m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp | ext_2020 最佳参数: {'bootstrap': False, 'max_depth': 45, 'max_features': 0.425643182642499, 'min_samples_leaf': 1, 'min_samples_split': 4, 'n_estimators': 297} | R²_train=0.999 | R²_test=0.914
Fitting 20 folds for each of 200 candidates, totalling 4000 fits
✅ city2020_lst_ratio_grid_1080m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp | hr_2020 最佳参数: {'bootstrap': True, 'max_depth': 38, 'max_features': 0.34902953310125123, 'min_samples_leaf': 1, 'min_samples_split': 3, 'n_estimators': 637} | R²_train=0.954 | R²_test=0.728
Fitting 20 folds for each of 2